In [1]:
!wget https://raw.githubusercontent.com/jiangshdd/ReviewCritique/refs/heads/main/data/ReviewCritique.jsonl -O ReviewCritique.jsonl

--2026-05-14 16:29:05--  https://raw.githubusercontent.com/jiangshdd/ReviewCritique/refs/heads/main/data/ReviewCritique.jsonl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7444779 (7.1M) [text/plain]
Saving to: ‘ReviewCritique.jsonl’

ReviewCritique.json 100%[===================>]   7.10M  --.-KB/s    in 0.07s   

2026-05-14 16:29:06 (103 MB/s) - ‘ReviewCritique.jsonl’ saved [7444779/7444779]



In [2]:
import json
with open("ReviewCritique.jsonl", "r") as f:
    data = [json.loads(line) for line in f]

In [3]:
segs = []

for idx, item in enumerate(data):
    entry = []
    for key in item.keys():
        if key.startswith("review#"):
            for seg in item[key]['review']:
                entry.append([seg["segment_text"], seg["reliability"]])
    segs.append(entry)

In [4]:
from openai import OpenAI
import os

client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY")
)

In [5]:
filter_prompt = """
You are filtering pairs of peer review segments. For each pair, decide whether the two segments make substantively the same claim or critique about the paper.

Return TRUE only if both segments express the same underlying point (same critique, same observation, same summary content) with same direction of comment. Return FALSE if:
- Either segment is a section header or boilerplate (e.g., "Clarity:", "Strengths:", "Novelty And Reproducibility:")
- The segments are topically related but make different claims (e.g., one says the paper is about few-shot, the other zero-shot)
- The segments share vocabulary but argue different points
- One is a question and the other is a statement on the same topic but not equivalent in meaning

Input format: list of (index, segment_a, segment_b)
Output format: JSON list of indices where the answer is TRUE. Only the list, no explanation.

Example input:
[(0, "Clarity:", "Clarity, Quality, Novelty:"),
 (1, "The paper proposes a few-shot method.", "The paper focuses on zero-shot prompting."),
 (2, "Results show 6B model matches 175B.", "Smaller models reach larger model performance.")]

Example output:
[2]

"""

In [6]:
from pydantic import BaseModel
class Response(BaseModel):
    idx: list[int]

def filter(matches):
    pairs = []
    new_idx_to_old = {}
    idx = 0
    for i, match in enumerate(matches):
        if len(match) > 0:
            pairs.append((idx, match[0], match[1]))
            new_idx_to_old[idx] = i
            idx += 1 

    response = client.chat.completions.parse(
        model="openai/gpt-5.5",
        messages=[
                    {
                        "role": "system",
                        "content": filter_prompt
                    },
                    {
                        "role": "user",
                        "content": str(pairs)
                    }
                ],
        extra_body={"reasoning": {"enabled": False}},
        response_format=Response
    )

    true_indices = response.choices[0].message.parsed.idx
    return [matches[new_idx_to_old[idx]] for idx in true_indices]

In [7]:
import tqdm
import numpy as np
import pylab
from sklearn.metrics import f1_score

all_matches = []
for paper in tqdm.tqdm(segs):
    if len(paper) == 0:
        all_matches.append({})
        continue
    inputs = [str(seg[0]) for seg in paper]
    embeddings = []
    for i in range(0, len(inputs), 64):
        batch = inputs[i:i+64]
        _embeddings = client.embeddings.create(
            model="google/gemini-embedding-2-preview",
            input=batch,
            encoding_format="float"
        )
        _embeddings = [_embeddings.data[i].embedding for i in range(len(_embeddings.data))]
        embeddings.extend(_embeddings)
    embeddings = np.array(embeddings)
    sim = np.dot(embeddings, embeddings.T)
    sim[np.arange(sim.shape[0]), np.arange(sim.shape[0])] = 0
    matches = []
    for i in range(sim.shape[0]):
        similar_item = []
        for j in range(i+1, sim.shape[0]):
            if sim[i][j] > 0.8:
                if len(paper[i][0]) > 30 and len(paper[j][0]) > 30:
                    similar_item.append(
                        (paper[i][0], paper[j][0], paper[i][1], paper[j][1], sim[i][j])
                    ) 
        if len(similar_item) > 0:
            matches.extend(similar_item)
    
    matches = filter(matches)
    all_matches.extend(matches) 
    print(f1_score([match[2] == "No" for match in all_matches if len(match) > 0], [match[3] == "No" for match in all_matches if len(match) > 0], zero_division=np.nan))


  1%|          | 1/100 [00:07<11:55,  7.23s/it]

0.0


  2%|▏         | 2/100 [00:13<10:51,  6.65s/it]

0.2857142857142857


  3%|▎         | 3/100 [00:17<08:42,  5.38s/it]

...

0.34782608695652173


 99%|█████████▉| 99/100 [10:03<00:05,  5.20s/it]

0.3446153846153846


100%|██████████| 100/100 [10:09<00:00,  6.10s/it]

0.3525835866261398


In [8]:
def compute_metrics(all_matches):
    pairs = [m for m in all_matches if len(m) > 0]
    y_true = [m[2] for m in pairs]
    y_pred = [m[3] for m in pairs]

    def metrics_for(positive):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == positive and p == positive)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != positive and p == positive)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == positive and p != positive)
        tn = sum(1 for t, p in zip(y_true, y_pred) if t != positive and p != positive)
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        selectivity = tn / (tn + fp) if (tn + fp) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        return {
            "positive_label": "reliable" if positive == "Yes" else "not reliable",
            "true_positive": tp,
            "false_positive": fp,
            "false_negative": fn,
            "precision": precision,
            "recall": recall,
            "sensitivity": recall,
            "selectivity": selectivity,
            "f1": f1,
        }

    return {
        "reliable_as_positive": metrics_for("Yes"),
        "not_reliable_as_positive": metrics_for("No"),
    }

In [9]:
compute_metrics(all_matches)

{'reliable_as_positive': {'positive_label': 'reliable',
  'true_positive': 2049,
  'false_positive': 121,
  'false_negative': 92,
  'precision': 0.9442396313364055,
  'recall': 0.9570294255021018,
  'sensitivity': 0.9570294255021018,
  'selectivity': 0.3240223463687151,
  'f1': 0.9505915100904663},
 'not_reliable_as_positive': {'positive_label': 'not reliable',
  'true_positive': 58,
  'false_positive': 92,
  'false_negative': 121,
  'precision': 0.38666666666666666,
  'recall': 0.3240223463687151,
  'sensitivity': 0.3240223463687151,
  'selectivity': 0.9570294255021018,
  'f1': 0.3525835866261398}}